
# Cleaning Dataset Biofarmaka — AgriDash Kabupaten Kediri

Notebook ini digunakan untuk melakukan **cleaning dan standardisasi dataset Biofarmaka BPS** untuk kebutuhan AgriDash.

## Aturan cleaning yang digunakan

- Tanda `-` pada dataset BPS dianggap **NULL / data tidak tersedia**, sehingga dikonversi menjadi `NaN`.
- Tanda `...` juga diperlakukan sebagai `NaN`.
- Nilai `0` yang memang tertulis sebagai angka `0` tetap dipertahankan sebagai **0**, karena berbeda dengan `-`.
- Baris `Kediri` pada dataset per kecamatan dianggap sebagai **total Kabupaten Kediri**, bukan kecamatan ke-27.
- `Kayen Kidul` dinormalisasi menjadi `Kayenkidul` agar konsisten dengan master kecamatan AgriDash.
- Data asli tingkat kecamatan **tidak dianggap sebagai estimasi**, sehingga `is_estimate = False`.
- Data luas panen tetap mempertahankan satuan dari sumber dan tidak dikonversi secara paksa.
- Hasil cleaning hanya disimpan sebagai **dataset clean**, sedangkan hasil validasi hanya ditampilkan di notebook dan **tidak disimpan sebagai CSV**.


In [1]:

import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)



## 1. Menentukan lokasi project

Path dibuat menggunakan `os` agar notebook dapat dijalankan dari repository GitHub tanpa menggunakan path Windows yang hardcode.


In [2]:

def cari_project_root(start_path=None):
    start = os.path.abspath(start_path or os.getcwd())
    current = start

    while True:
        if (
            os.path.isdir(os.path.join(current, "data"))
            and (
                os.path.isdir(os.path.join(current, "notebooks"))
                or os.path.isdir(os.path.join(current, ".git"))
            )
        ):
            return current

        parent = os.path.dirname(current)

        if parent == current:
            break

        current = parent

    return os.path.abspath(os.path.join(start, ".."))


PROJECT_ROOT = cari_project_root()

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_BIOFARMAKA = os.path.join(DATA_DIR, "raw", "biofarmaka")

# Output hasil cleaning
OUT_BIOFARMAKA = os.path.join(
    DATA_DIR,
    "clean",
    "biofarmaka"
)

os.makedirs(OUT_BIOFARMAKA, exist_ok=True)

print("PROJECT ROOT :", PROJECT_ROOT)
print("RAW BIO", RAW_BIOFARMAKA)
print("CLEAN OUTPUT :", OUT_BIOFARMAKA)


PROJECT ROOT : d:\SDT\Project Analitika Data\AgriDash-0.1
RAW BIO d:\SDT\Project Analitika Data\AgriDash-0.1\data\raw\biofarmaka
CLEAN OUTPUT : d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\biofarmaka



## 2. Nama file sumber

Pastikan tiga file berikut berada di:

`data/raw/hortikultura/`


In [3]:

FILE_LUAS_KAB = "Luas_Panen_Biofarmaka.csv"
FILE_LUAS_KEC = "Luas_Panen_Biofarmaka_per_kecamatan.csv"
FILE_PRODUKSI_KEC = "Produksi_Biofarmaka_per_kecamatan.csv"

PATH_LUAS_KAB = os.path.join(RAW_BIOFARMAKA,  FILE_LUAS_KAB)
PATH_LUAS_KEC = os.path.join(RAW_BIOFARMAKA,  FILE_LUAS_KEC)
PATH_PRODUKSI_KEC = os.path.join(RAW_BIOFARMAKA, FILE_PRODUKSI_KEC)

paths = {
    "Luas Panen Kabupaten": PATH_LUAS_KAB,
    "Luas Panen Kecamatan": PATH_LUAS_KEC,
    "Produksi Kecamatan": PATH_PRODUKSI_KEC
}

for nama, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File {nama} tidak ditemukan:\n{path}"
        )

print("Semua file sumber ditemukan.")


FileNotFoundError: File Produksi Kecamatan tidak ditemukan:
d:\SDT\Project Analitika Data\AgriDash-0.1\data\raw\biofarmaka\Produksi_Biofarmaka_per_kecamatan.csv

In [ ]:

# Membaca dataset mentah

df_luas_kab_raw = pd.read_csv(
    PATH_LUAS_KAB,
    encoding="utf-8-sig"
)

df_luas_kec_raw = pd.read_csv(
    PATH_LUAS_KEC,
    encoding="utf-8-sig"
)

df_produksi_kec_raw = pd.read_csv(
    PATH_PRODUKSI_KEC,
    encoding="utf-8-sig"
)

print("Luas Panen Kabupaten :", df_luas_kab_raw.shape)
print("Luas Panen Kecamatan:", df_luas_kec_raw.shape)
print("Produksi Kecamatan  :", df_produksi_kec_raw.shape)


Luas Panen Kabupaten : (19, 2)
Luas Panen Kecamatan: (27, 17)
Produksi Kecamatan  : (27, 17)



## 3. Melihat struktur data sebelum cleaning


In [ ]:

print("=== LUAS PANEN KABUPATEN ===")
display(df_luas_kab_raw.head())

print("\n=== LUAS PANEN PER KECAMATAN ===")
display(df_luas_kec_raw.head())

print("\n=== PRODUKSI PER KECAMATAN ===")
display(df_produksi_kec_raw.head())


=== LUAS PANEN KABUPATEN ===


,Jenis Tanaman,Luas Panen Tanaman Biofarmaka
0,Jahe/Ginger(m2),92260
1,Kapulaga/Java Cardamom(m2),5505
2,Keji Beling/Verbenaceae(m2),...
3,Kencur/East Indian Galangal(m2),1027
4,Kunyit/Turmeric(m2),4476188



=== LUAS PANEN PER KECAMATAN ===


,Kecamatan,Luas Panen Jahe (meter persegi) (M2),Luas Panen Kapulaga (meter persegi) (M2),Luas Panen Keji Beling (meter persegi) (M2),Luas Panen Kencur (meter persegi) (M2),Luas Panen Kunyit (meter persegi) (M2),Luas Panen Laos/Lengkuas (meter persegi) (M2),Luas Panen Lempuyang (meter persegi) (M2),Luas Panen Lidah Buaya (meter persegi) (M2),Luas Panen Mahkota Dewa (pohon) (Pohon),Luas Panen Mengkudu/Pace (pohon) (Pohon),Luas Panen Sambiloto (meter persegi) (M2),Luas Panen Temuireng (meter persegi) (M2),Luas Panen Temukunci (meter persegi) (M2),Luas Panen Temulawak (meter persegi) (M2),Luas Panen Jeruk Nipis (pohon) (Pohon),Luas Panen Serai (meter persegi) (Pohon)
0,Mojo,5000,5500,...,-,1000,1000,-,-,-,-,-,-,-,4000,500,3000
1,Semen,-,-,...,-,-,-,-,-,-,-,-,-,-,-,15000,5000
2,Ngadiluwih,-,-,...,-,-,-,-,-,-,-,-,-,-,-,-,-
3,Kras,-,-,...,-,-,-,-,-,-,-,-,-,-,-,300,-
4,Ringinrejo,-,-,...,-,-,-,-,-,-,-,-,-,-,-,196,-



=== PRODUKSI PER KECAMATAN ===


,Kecamatan,Produksi Jahe (kilogram) (Kg),Produksi Kapulaga (kilogram) (Kg),Produksi Keji Beling (kilogram) (Kg),Produksi Kencur (kilogram) (Kg),Produksi Kunyit (kilogram) (Kg),Produksi Laos/Lengkuas (kilogram) (Kg),Produksi Lempuyang (kilogram) (Kg),Produksi Lidah Buaya (kilogram) (Kg),Produksi Mahkota Dewa (kilogram) (Kg),Produksi Mengkudu/Pace (kilogram) (Kg),Produksi Sambiloto (kilogram) (Kg),Produksi Temuireng (kilogram) (Kg),Produksi Temukunci (kilogram) (Kg),Produksi Temulawak (kilogram) (Kg),Produksi Jeruk Nipis (kilogram) (Kg),Produksi Serai (kilogram) (Kg)
0,Mojo,9200,21250,...,-,8000,7000,-,-,-,-,-,-,-,8500,33000,40500
1,Semen,-,-,...,-,-,-,-,-,-,-,-,-,-,-,303000,20000
2,Ngadiluwih,-,-,...,-,-,-,-,-,-,-,-,-,-,-,-,-
3,Kras,-,-,...,-,-,-,-,-,-,-,-,-,-,-,7050,-
4,Ringinrejo,-,-,...,-,-,-,-,-,-,-,-,-,-,-,4120,-



## 4. Master kecamatan

Kabupaten Kediri memiliki 26 kecamatan. Baris `Kediri` yang muncul pada dataset BPS akan dipisahkan sebagai total Kabupaten.


In [ ]:

MASTER_KECAMATAN = [
    "Mojo",
    "Semen",
    "Ngadiluwih",
    "Kras",
    "Ringinrejo",
    "Kandat",
    "Wates",
    "Ngancar",
    "Plosoklaten",
    "Gurah",
    "Puncu",
    "Kepung",
    "Kandangan",
    "Pare",
    "Badas",
    "Kunjang",
    "Plemahan",
    "Purwoasri",
    "Papar",
    "Pagu",
    "Kayenkidul",
    "Gampengrejo",
    "Ngasem",
    "Banyakan",
    "Grogol",
    "Tarokan"
]

NORMALISASI_KECAMATAN = {
    "Kayen Kidul": "Kayenkidul"
}


def normalisasi_kecamatan(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    return NORMALISASI_KECAMATAN.get(x, x)


df_luas_kec_raw["Kecamatan"] = (
    df_luas_kec_raw["Kecamatan"]
    .map(normalisasi_kecamatan)
)

df_produksi_kec_raw["Kecamatan"] = (
    df_produksi_kec_raw["Kecamatan"]
    .map(normalisasi_kecamatan)
)


print("Jumlah master kecamatan:", len(MASTER_KECAMATAN))


Jumlah master kecamatan: 26



## 5. Fungsi cleaning nilai

**Penting:** berdasarkan keterangan BPS yang digunakan pada dataset ini:

- `-` = **NULL**
- `...` = **NULL**
- angka `0` = tetap **0**

Jadi `-` **tidak boleh diubah menjadi 0** karena akan membuat data kosong dianggap sebagai produksi/luas panen nol.


In [ ]:

def bersihkan_nilai(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Kode data kosong dari sumber BPS
    if s in ["-", "..."]:
        return np.nan

    # Hilangkan pemisah ribuan jika ada
    s = s.replace(",", "")

    return pd.to_numeric(s, errors="coerce")



## 6. Cleaning luas panen per kecamatan

Format wide dari BPS diubah menjadi format long agar lebih mudah digunakan untuk database, analisis, dashboard, dan integrasi dengan dataset AgriDash lainnya.


In [ ]:

def ekstrak_nama_komoditas_luas(kolom):
    nama = re.sub(
        r"^Luas Panen\s+",
        "",
        str(kolom),
        flags=re.IGNORECASE
    )

    # Menghapus informasi satuan di dalam tanda kurung
    nama = re.sub(r"\s*\(.*$", "", nama).strip()

    return nama


def ekstrak_satuan_luas(kolom):
    lower = str(kolom).lower()

    if "pohon" in lower:
        return "Pohon"

    if "meter persegi" in lower or "(m2)" in lower:
        return "m2"

    return "Tidak diketahui"


def wide_ke_long_luas(df_raw):
    records = []

    for _, row in df_raw.iterrows():
        kecamatan = normalisasi_kecamatan(row["Kecamatan"])

        for kolom in df_raw.columns[1:]:
            records.append({
                "kecamatan": kecamatan,
                "komoditas": ekstrak_nama_komoditas_luas(kolom),
                "luas_panen": bersihkan_nilai(row[kolom]),
                "satuan_luas_panen": ekstrak_satuan_luas(kolom)
            })

    return pd.DataFrame(records)


df_luas_long = wide_ke_long_luas(df_luas_kec_raw)

# Data asli BPS tingkat kecamatan, bukan hasil estimasi
df_luas_long["is_estimate"] = False
df_luas_long["metode"] = "Data sumber BPS tingkat kecamatan"

# Status kualitas
df_luas_long["status_qc"] = np.where(
    df_luas_long["luas_panen"].isna(),
    "NULL",
    "OK"
)

display(df_luas_long.head(20))


,kecamatan,komoditas,luas_panen,satuan_luas_panen,is_estimate,metode,status_qc
0,Mojo,Jahe,5000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
1,Mojo,Kapulaga,5500.0,m2,False,Data sumber BPS tingkat kecamatan,OK
2,Mojo,Keji Beling,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
3,Mojo,Kencur,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
4,Mojo,Kunyit,1000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
5,Mojo,Laos/Lengkuas,1000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
6,Mojo,Lempuyang,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
7,Mojo,Lidah Buaya,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
8,Mojo,Mahkota Dewa,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL
9,Mojo,Mengkudu/Pace,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL



## 7. Cleaning produksi per kecamatan


In [ ]:

def ekstrak_nama_komoditas_produksi(kolom):
    nama = re.sub(
        r"^Produksi\s+",
        "",
        str(kolom),
        flags=re.IGNORECASE
    )

    nama = re.sub(r"\s*\(.*$", "", nama).strip()

    return nama


def wide_ke_long_produksi(df_raw):
    records = []

    for _, row in df_raw.iterrows():
        kecamatan = normalisasi_kecamatan(row["Kecamatan"])

        for kolom in df_raw.columns[1:]:
            records.append({
                "kecamatan": kecamatan,
                "komoditas": ekstrak_nama_komoditas_produksi(kolom),
                "produksi_kg": bersihkan_nilai(row[kolom])
            })

    return pd.DataFrame(records)


df_produksi_long = wide_ke_long_produksi(
    df_produksi_kec_raw
)

df_produksi_long["is_estimate"] = False
df_produksi_long["metode"] = "Data sumber BPS tingkat kecamatan"

df_produksi_long["status_qc"] = np.where(
    df_produksi_long["produksi_kg"].isna(),
    "NULL",
    "OK"
)

display(df_produksi_long.head(20))


,kecamatan,komoditas,produksi_kg,is_estimate,metode,status_qc
0,Mojo,Jahe,9200.0,False,Data sumber BPS tingkat kecamatan,OK
1,Mojo,Kapulaga,21250.0,False,Data sumber BPS tingkat kecamatan,OK
2,Mojo,Keji Beling,NaN,False,Data sumber BPS tingkat kecamatan,NULL
3,Mojo,Kencur,NaN,False,Data sumber BPS tingkat kecamatan,NULL
4,Mojo,Kunyit,8000.0,False,Data sumber BPS tingkat kecamatan,OK
5,Mojo,Laos/Lengkuas,7000.0,False,Data sumber BPS tingkat kecamatan,OK
6,Mojo,Lempuyang,NaN,False,Data sumber BPS tingkat kecamatan,NULL
7,Mojo,Lidah Buaya,NaN,False,Data sumber BPS tingkat kecamatan,NULL
8,Mojo,Mahkota Dewa,NaN,False,Data sumber BPS tingkat kecamatan,NULL
9,Mojo,Mengkudu/Pace,NaN,False,Data sumber BPS tingkat kecamatan,NULL



## 8. Memisahkan 26 kecamatan dan total Kabupaten

Baris `Kediri` tidak dimasukkan ke dataset kecamatan karena merupakan total Kabupaten.


In [ ]:

# LUAS PANEN
df_luas_kec_clean = df_luas_long[
    df_luas_long["kecamatan"] != "Kediri"
].copy()

df_luas_kab_clean = df_luas_long[
    df_luas_long["kecamatan"] == "Kediri"
].copy()


# PRODUKSI
df_produksi_kec_clean = df_produksi_long[
    df_produksi_long["kecamatan"] != "Kediri"
].copy()

df_produksi_kab_clean = df_produksi_long[
    df_produksi_long["kecamatan"] == "Kediri"
].copy()


print("Jumlah kecamatan pada luas panen:",
      df_luas_kec_clean["kecamatan"].nunique())

print("Jumlah kecamatan pada produksi:",
      df_produksi_kec_clean["kecamatan"].nunique())


Jumlah kecamatan pada luas panen: 26
Jumlah kecamatan pada produksi: 26



## 9. Validasi hasil cleaning

Validasi **hanya ditampilkan di notebook** dan tidak disimpan sebagai file.

Pengecekan meliputi:
- apakah terdapat 26 kecamatan;
- apakah ada kecamatan yang hilang;
- apakah ada duplikasi kecamatan-komoditas;
- jumlah NULL;
- nilai negatif;
- perbandingan total 26 kecamatan dengan total Kabupaten dari BPS.

Perbedaan total **tidak otomatis berarti cleaning salah**, terutama jika sumber memiliki NULL atau terdapat perbedaan cakupan data.


In [ ]:

def tampilkan_validasi_kecamatan(df, nama):
    daftar_data = set(
        df["kecamatan"].dropna().unique()
    )

    master = set(MASTER_KECAMATAN)

    print(f"=== VALIDASI {nama} ===")
    print("Jumlah kecamatan :", len(daftar_data))

    print(
        "Kecamatan yang hilang:",
        sorted(master - daftar_data)
    )

    print(
        "Nama kecamatan di luar master:",
        sorted(daftar_data - master)
    )

    duplikat = df.duplicated(
        subset=["kecamatan", "komoditas"]
    ).sum()

    print(
        "Duplikasi kecamatan-komoditas:",
        duplikat
    )

    print(
        "Jumlah NULL:",
        df["luas_panen"].isna().sum()
        if "luas_panen" in df.columns
        else df["produksi_kg"].isna().sum()
    )


tampilkan_validasi_kecamatan(
    df_luas_kec_clean,
    "LUAS PANEN"
)

print()

tampilkan_validasi_kecamatan(
    df_produksi_kec_clean,
    "PRODUKSI"
)


=== VALIDASI LUAS PANEN ===
Jumlah kecamatan : 26
Kecamatan yang hilang: []
Nama kecamatan di luar master: []
Duplikasi kecamatan-komoditas: 0
Jumlah NULL: 366

=== VALIDASI PRODUKSI ===
Jumlah kecamatan : 26
Kecamatan yang hilang: []
Nama kecamatan di luar master: []
Duplikasi kecamatan-komoditas: 0
Jumlah NULL: 366


In [ ]:

def validasi_total(df_kec, df_kab, kolom_nilai, nama_nilai):
    hasil = []

    komoditas_semua = sorted(
        set(df_kec["komoditas"].dropna())
        |
        set(df_kab["komoditas"].dropna())
    )

    for komoditas in komoditas_semua:

        nilai_kec = df_kec.loc[
            df_kec["komoditas"] == komoditas,
            kolom_nilai
        ]

        nilai_kab = df_kab.loc[
            df_kab["komoditas"] == komoditas,
            kolom_nilai
        ]

        total_kec = nilai_kec.sum(min_count=1)

        total_kab = (
            nilai_kab.iloc[0]
            if len(nilai_kab) > 0
            else np.nan
        )

        if pd.isna(total_kec) or pd.isna(total_kab):
            status = "TIDAK DAPAT DIVALIDASI"
            selisih = np.nan

        else:
            selisih = total_kec - total_kab

            status = (
                "SESUAI"
                if np.isclose(
                    total_kec,
                    total_kab,
                    atol=0.01
                )
                else "BERBEDA"
            )

        hasil.append({
            "komoditas": komoditas,
            "total_26_kecamatan": total_kec,
            "total_kabupaten_bps": total_kab,
            "selisih": selisih,
            "status": status
        })

    return pd.DataFrame(hasil)


validasi_luas = validasi_total(
    df_luas_kec_clean,
    df_luas_kab_clean,
    "luas_panen",
    "luas_panen"
)

validasi_produksi = validasi_total(
    df_produksi_kec_clean,
    df_produksi_kab_clean,
    "produksi_kg",
    "produksi_kg"
)

print("=== VALIDASI TOTAL LUAS PANEN ===")
display(validasi_luas)

print("=== VALIDASI TOTAL PRODUKSI ===")
display(validasi_produksi)


=== VALIDASI TOTAL LUAS PANEN ===


,komoditas,total_26_kecamatan,total_kabupaten_bps,selisih,status
0,Jahe,92260.0,92260.0,0.0,SESUAI
1,Jeruk Nipis,18823.0,18823.0,0.0,SESUAI
2,Kapulaga,5505.0,5505.0,0.0,SESUAI
3,Keji Beling,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI
4,Kencur,1027.0,1027.0,0.0,SESUAI
5,Kunyit,4476188.0,4476188.0,0.0,SESUAI
6,Laos/Lengkuas,3504.0,3504.0,0.0,SESUAI
7,Lempuyang,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI
8,Lidah Buaya,1650.0,1650.0,0.0,SESUAI
9,Mahkota Dewa,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI


=== VALIDASI TOTAL PRODUKSI ===


,komoditas,total_26_kecamatan,total_kabupaten_bps,selisih,status
0,Jahe,133603.0,133603.0,0.0,SESUAI
1,Jeruk Nipis,374273.0,374273.0,0.0,SESUAI
2,Kapulaga,21263.0,21263.0,0.0,SESUAI
3,Keji Beling,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI
4,Kencur,6910.0,6910.0,0.0,SESUAI
5,Kunyit,6045767.0,6045767.0,0.0,SESUAI
6,Laos/Lengkuas,11444.0,11444.0,0.0,SESUAI
7,Lempuyang,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI
8,Lidah Buaya,12595.0,12595.0,0.0,SESUAI
9,Mahkota Dewa,NaN,NaN,NaN,TIDAK DAPAT DIVALIDASI



## 10. Pemeriksaan nilai NULL

Karena `-` dari BPS berarti NULL, bagian ini memperlihatkan komoditas dan kecamatan yang memiliki data kosong.

**NULL tidak diganti dengan 0.**


In [ ]:

print("=== NULL PADA LUAS PANEN ===")

display(
    df_luas_kec_clean[
        df_luas_kec_clean["luas_panen"].isna()
    ].sort_values(
        ["kecamatan", "komoditas"]
    )
)

print("=== NULL PADA PRODUKSI ===")

display(
    df_produksi_kec_clean[
        df_produksi_kec_clean["produksi_kg"].isna()
    ].sort_values(
        ["kecamatan", "komoditas"]
    )
)


=== NULL PADA LUAS PANEN ===


,kecamatan,komoditas,luas_panen,satuan_luas_panen,is_estimate,metode,status_qc
224,Badas,Jahe,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
238,Badas,Jeruk Nipis,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL
225,Badas,Kapulaga,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
226,Badas,Keji Beling,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
227,Badas,Kencur,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
...,...,...,...,...,...,...,...
106,Wates,Sambiloto,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
111,Wates,Serai,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL
107,Wates,Temuireng,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
108,Wates,Temukunci,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL


=== NULL PADA PRODUKSI ===


,kecamatan,komoditas,produksi_kg,is_estimate,metode,status_qc
224,Badas,Jahe,NaN,False,Data sumber BPS tingkat kecamatan,NULL
238,Badas,Jeruk Nipis,NaN,False,Data sumber BPS tingkat kecamatan,NULL
225,Badas,Kapulaga,NaN,False,Data sumber BPS tingkat kecamatan,NULL
226,Badas,Keji Beling,NaN,False,Data sumber BPS tingkat kecamatan,NULL
227,Badas,Kencur,NaN,False,Data sumber BPS tingkat kecamatan,NULL
...,...,...,...,...,...,...
106,Wates,Sambiloto,NaN,False,Data sumber BPS tingkat kecamatan,NULL
111,Wates,Serai,NaN,False,Data sumber BPS tingkat kecamatan,NULL
107,Wates,Temuireng,NaN,False,Data sumber BPS tingkat kecamatan,NULL
108,Wates,Temukunci,NaN,False,Data sumber BPS tingkat kecamatan,NULL



## 11. Pemeriksaan satuan luas

Satuan tidak dikonversi secara otomatis. Ini penting karena dataset biofarmaka dapat memiliki komoditas yang dilaporkan dalam `m2` maupun `Pohon`.

Jika terdapat ketidaksesuaian satuan pada sumber, data ditandai untuk pemeriksaan lebih lanjut.


In [ ]:

print("Distribusi satuan luas panen:")
display(
    df_luas_kec_clean[
        ["komoditas", "satuan_luas_panen"]
    ]
    .drop_duplicates()
    .sort_values("komoditas")
)


Distribusi satuan luas panen:


,komoditas,satuan_luas_panen
0,Jahe,m2
14,Jeruk Nipis,Pohon
1,Kapulaga,m2
2,Keji Beling,m2
3,Kencur,m2
4,Kunyit,m2
5,Laos/Lengkuas,m2
6,Lempuyang,m2
7,Lidah Buaya,m2
8,Mahkota Dewa,Pohon



## 12. Menyiapkan dataset final

Dataset final yang disimpan hanya dataset hasil cleaning. **File validasi tidak disimpan.**

Output:
- `luas_panen_biofarmaka_per_kecamatan.csv`
- `luas_panen_biofarmaka_kabupaten.csv`
- `produksi_biofarmaka_per_kecamatan.csv`
- `produksi_biofarmaka_kabupaten.csv`


In [ ]:

# Pembulatan angka
for df in [
    df_luas_kec_clean,
    df_luas_kab_clean
]:
    df["luas_panen"] = df["luas_panen"].round(2)


for df in [
    df_produksi_kec_clean,
    df_produksi_kab_clean
]:
    df["produksi_kg"] = df["produksi_kg"].round(2)


# Pastikan kolom berada pada urutan yang rapi
df_luas_kec_clean = df_luas_kec_clean[
    [
        "kecamatan",
        "komoditas",
        "luas_panen",
        "satuan_luas_panen",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_luas_kab_clean = df_luas_kab_clean[
    [
        "kecamatan",
        "komoditas",
        "luas_panen",
        "satuan_luas_panen",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_produksi_kec_clean = df_produksi_kec_clean[
    [
        "kecamatan",
        "komoditas",
        "produksi_kg",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_produksi_kab_clean = df_produksi_kab_clean[
    [
        "kecamatan",
        "komoditas",
        "produksi_kg",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]


In [ ]:

# Nama file output
OUT_LUAS_KEC = os.path.join(
    OUT_BIOFARMAKA,
    "luas_panen_biofarmaka_per_kecamatan.csv"
)

OUT_LUAS_KAB = os.path.join(
    OUT_BIOFARMAKA,
    "luas_panen_biofarmaka_kabupaten.csv"
)

OUT_PROD_KEC = os.path.join(
    OUT_BIOFARMAKA,
    "produksi_biofarmaka_per_kecamatan.csv"
)

OUT_PROD_KAB = os.path.join(
    OUT_BIOFARMAKA,
    "produksi_biofarmaka_kabupaten.csv"
)


# Simpan dataset clean
df_luas_kec_clean.to_csv(
    OUT_LUAS_KEC,
    index=False,
    encoding="utf-8-sig"
)

df_luas_kab_clean.to_csv(
    OUT_LUAS_KAB,
    index=False,
    encoding="utf-8-sig"
)

df_produksi_kec_clean.to_csv(
    OUT_PROD_KEC,
    index=False,
    encoding="utf-8-sig"
)

df_produksi_kab_clean.to_csv(
    OUT_PROD_KAB,
    index=False,
    encoding="utf-8-sig"
)


print("Dataset clean berhasil disimpan ke:")
print(OUT_BIOFARMAKA)

print("\nFile output:")
print("-", os.path.basename(OUT_LUAS_KEC))
print("-", os.path.basename(OUT_LUAS_KAB))
print("-", os.path.basename(OUT_PROD_KEC))
print("-", os.path.basename(OUT_PROD_KAB))


Dataset clean berhasil disimpan ke:
d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\biofarmaka

File output:
- luas_panen_biofarmaka_per_kecamatan.csv
- luas_panen_biofarmaka_kabupaten.csv
- produksi_biofarmaka_per_kecamatan.csv
- produksi_biofarmaka_kabupaten.csv



## 13. Preview dataset final


In [ ]:

print("=== LUAS PANEN PER KECAMATAN ===")
display(df_luas_kec_clean.head(20))

print("=== PRODUKSI PER KECAMATAN ===")
display(df_produksi_kec_clean.head(20))

print("=== LUAS PANEN KABUPATEN ===")
display(df_luas_kab_clean)

print("=== PRODUKSI KABUPATEN ===")
display(df_produksi_kab_clean)


=== LUAS PANEN PER KECAMATAN ===


,kecamatan,komoditas,luas_panen,satuan_luas_panen,is_estimate,metode,status_qc
0,Mojo,Jahe,5000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
1,Mojo,Kapulaga,5500.0,m2,False,Data sumber BPS tingkat kecamatan,OK
2,Mojo,Keji Beling,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
3,Mojo,Kencur,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
4,Mojo,Kunyit,1000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
5,Mojo,Laos/Lengkuas,1000.0,m2,False,Data sumber BPS tingkat kecamatan,OK
6,Mojo,Lempuyang,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
7,Mojo,Lidah Buaya,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
8,Mojo,Mahkota Dewa,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL
9,Mojo,Mengkudu/Pace,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL


=== PRODUKSI PER KECAMATAN ===


,kecamatan,komoditas,produksi_kg,is_estimate,metode,status_qc
0,Mojo,Jahe,9200.0,False,Data sumber BPS tingkat kecamatan,OK
1,Mojo,Kapulaga,21250.0,False,Data sumber BPS tingkat kecamatan,OK
2,Mojo,Keji Beling,NaN,False,Data sumber BPS tingkat kecamatan,NULL
3,Mojo,Kencur,NaN,False,Data sumber BPS tingkat kecamatan,NULL
4,Mojo,Kunyit,8000.0,False,Data sumber BPS tingkat kecamatan,OK
5,Mojo,Laos/Lengkuas,7000.0,False,Data sumber BPS tingkat kecamatan,OK
6,Mojo,Lempuyang,NaN,False,Data sumber BPS tingkat kecamatan,NULL
7,Mojo,Lidah Buaya,NaN,False,Data sumber BPS tingkat kecamatan,NULL
8,Mojo,Mahkota Dewa,NaN,False,Data sumber BPS tingkat kecamatan,NULL
9,Mojo,Mengkudu/Pace,NaN,False,Data sumber BPS tingkat kecamatan,NULL


=== LUAS PANEN KABUPATEN ===


,kecamatan,komoditas,luas_panen,satuan_luas_panen,is_estimate,metode,status_qc
416,Kediri,Jahe,92260.0,m2,False,Data sumber BPS tingkat kecamatan,OK
417,Kediri,Kapulaga,5505.0,m2,False,Data sumber BPS tingkat kecamatan,OK
418,Kediri,Keji Beling,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
419,Kediri,Kencur,1027.0,m2,False,Data sumber BPS tingkat kecamatan,OK
420,Kediri,Kunyit,4476188.0,m2,False,Data sumber BPS tingkat kecamatan,OK
421,Kediri,Laos/Lengkuas,3504.0,m2,False,Data sumber BPS tingkat kecamatan,OK
422,Kediri,Lempuyang,NaN,m2,False,Data sumber BPS tingkat kecamatan,NULL
423,Kediri,Lidah Buaya,1650.0,m2,False,Data sumber BPS tingkat kecamatan,OK
424,Kediri,Mahkota Dewa,NaN,Pohon,False,Data sumber BPS tingkat kecamatan,NULL
425,Kediri,Mengkudu/Pace,802.0,Pohon,False,Data sumber BPS tingkat kecamatan,OK


=== PRODUKSI KABUPATEN ===


,kecamatan,komoditas,produksi_kg,is_estimate,metode,status_qc
416,Kediri,Jahe,133603.0,False,Data sumber BPS tingkat kecamatan,OK
417,Kediri,Kapulaga,21263.0,False,Data sumber BPS tingkat kecamatan,OK
418,Kediri,Keji Beling,NaN,False,Data sumber BPS tingkat kecamatan,NULL
419,Kediri,Kencur,6910.0,False,Data sumber BPS tingkat kecamatan,OK
420,Kediri,Kunyit,6045767.0,False,Data sumber BPS tingkat kecamatan,OK
421,Kediri,Laos/Lengkuas,11444.0,False,Data sumber BPS tingkat kecamatan,OK
422,Kediri,Lempuyang,NaN,False,Data sumber BPS tingkat kecamatan,NULL
423,Kediri,Lidah Buaya,12595.0,False,Data sumber BPS tingkat kecamatan,OK
424,Kediri,Mahkota Dewa,NaN,False,Data sumber BPS tingkat kecamatan,NULL
425,Kediri,Mengkudu/Pace,6972.0,False,Data sumber BPS tingkat kecamatan,OK



# Kesimpulan

Dataset Biofarmaka telah dibersihkan dan distandarkan untuk kebutuhan AgriDash. Nilai `-` dan `...` diperlakukan sebagai **NULL (`NaN`)**, sedangkan angka `0` tetap dipertahankan sebagai nilai nol. Data `Kediri` dipisahkan sebagai total Kabupaten dan tidak dianggap sebagai kecamatan ke-27. Hasil validasi hanya ditampilkan di notebook sebagai kontrol kualitas dan tidak disimpan sebagai file. Dataset hasil cleaning disimpan pada `data/clean/hortikultura/biofarmaka/`.


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- 0. KONFIGURASI PATH
FILES = {
    "luas_nasional": "Luas_Panen_Biofarmaka.csv",
    "prod_nasional": "Total_Produksi_Biofarmaka.csv",
    "luas_kec": "Luas_Panen_Biofarmaka_per_kecamatan.csv",
    "prod_kec": "Total_Produksi_Biofarmaka_per_kecamatan.csv",
}

try:
    BASE = Path(__file__).resolve().parent      # dijalankan sebagai file .py
except NameError:
    BASE = Path.cwd()                           # dijalankan di notebook/interactive

def find_data_dir():
    kandidat = [BASE, BASE / "data", Path.cwd(), Path.cwd() / "data"]
    for d in kandidat:
        if all((d / f).exists() for f in FILES.values()):
            return d
    print("File CSV tidak ditemukan. Lokasi yang dicari:")
    for d in kandidat:
        print("  -", d)
    print("Isi folder skrip:", [p.name for p in BASE.iterdir()])
    raise FileNotFoundError("Taruh keempat CSV di folder yang sama dengan skrip, atau di subfolder 'data'.")

DATA_DIR = find_data_dir()
OUT_DIR = DATA_DIR / "output"
OUT_DIR.mkdir(exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

MISSING_TOKENS = ["-", "", " ", "NA", "N/A", "na", "null", "..."]
YEAR_MIN_VALID_RATIO = 0.30     # tahun dibuang jika <30% selnya terisi (2025 hanya ~14%)

# ---------------------------------------------------------------- 1. LOAD
def load_raw(path):
    # utf-8-sig membuang BOM; dtype=str agar '-' tidak merusak tipe kolom
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)

raw = {k: load_raw(DATA_DIR / v) for k, v in FILES.items()}

# ---------------------------------------------------------------- 2. AUDIT
def audit(df, name):
    m = df.isin(MISSING_TOKENS)
    print(f"\n=== AUDIT {name} | shape={df.shape} | duplikat={df.duplicated().sum()}")
    print("Missing token per tahun:\n", m.groupby(df["Tahun"]).sum().sum(axis=1).to_string())

for k, d in raw.items():
    audit(d, k)

# ---------------------------------------------------------------- 3. CLEAN HELPERS
def clean_text(s):
    return s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

def to_numeric(df, cols):
    out = df.copy()
    for c in cols:
        s = clean_text(out[c]).replace(MISSING_TOKENS, np.nan)
        s = s.str.replace(r"[^\d,.\-]", "", regex=True)   # buang karakter non-angka
        out[c] = pd.to_numeric(s, errors="coerce")
    return out

def clean_national(df, value_col, value_name):
    d = df.copy()
    d["Tahun"] = pd.to_numeric(d["Tahun"], errors="coerce").astype("Int64")
    d["Jenis Tanaman"] = clean_text(d["Jenis Tanaman"])
    # pisahkan nama tanaman dan satuan: 'Jahe/Ginger(m2)' -> Jahe | m2
    d["Satuan"] = d["Jenis Tanaman"].str.extract(r"\(([^)]*)\)\s*$")[0]
    d["Tanaman"] = (d["Jenis Tanaman"].str.replace(r"\([^)]*\)\s*$", "", regex=True)
                    .str.split("/").str[0].str.strip())
    d = to_numeric(d, [value_col]).rename(columns={value_col: value_name})
    d = d.drop(columns="Jenis Tanaman").drop_duplicates(["Tahun", "Tanaman"])
    return d[["Tahun", "Tanaman", "Satuan", value_name]]

def clean_kecamatan(df, value_name):
    d = df.copy()
    d["Tahun"] = pd.to_numeric(d["Tahun"], errors="coerce").astype("Int64")
    d["Kecamatan"] = clean_text(d["Kecamatan"]).str.title()
    val_cols = [c for c in d.columns if c not in ("Tahun", "Kecamatan")]
    d = to_numeric(d, val_cols)
    # wide -> long; nama tanaman diambil dari 'Luas Panen Jahe (..) (M2)'
    long = d.melt(["Tahun", "Kecamatan"], var_name="kolom", value_name=value_name)
    long["Tanaman"] = (long["kolom"]
                       .str.replace(r"^(Luas Panen|Produksi)\s+", "", regex=True)
                       .str.replace(r"\s*\(.*$", "", regex=True).str.strip())
    long["Satuan"] = long["kolom"].str.extract(r"\(([^()]*)\)\s*$")[0]
    return (long.drop(columns="kolom").drop_duplicates(["Tahun", "Kecamatan", "Tanaman"])
            [["Tahun", "Kecamatan", "Tanaman", "Satuan", value_name]])

nas_luas = clean_national(raw["luas_nasional"], "Luas Panen Tanaman Biofarmaka", "luas_panen")
nas_prod = clean_national(raw["prod_nasional"], "Produksi Tanaman Biofarmaka", "produksi_kg")
kec_luas = clean_kecamatan(raw["luas_kec"], "luas_panen")
kec_prod = clean_kecamatan(raw["prod_kec"], "produksi_kg")

# ---------------------------------------------------------------- 4. MERGE
# Satuan luas dipertahankan: m2 (rimpang/herba) vs pohon
kec = (kec_luas.rename(columns={"Satuan": "satuan_luas"})
       .merge(kec_prod.drop(columns="Satuan"), on=["Tahun", "Kecamatan", "Tanaman"],
              how="outer", validate="1:1"))
nas = (nas_luas.rename(columns={"Satuan": "satuan_luas"})
       .merge(nas_prod.drop(columns="Satuan"), on=["Tahun", "Tanaman"],
              how="outer", validate="1:1"))

# ---------------------------------------------------------------- 5. TAHUN KOSONG
def drop_empty_years(df, cols, name):
    ratio = df.groupby("Tahun")[cols].apply(lambda x: x.notna().mean().mean())
    print(f"\n[{name}] rasio data terisi per tahun:\n{ratio.round(3).to_string()}")
    bad = ratio[ratio < YEAR_MIN_VALID_RATIO].index
    print(f"-> tahun dibuang: {[int(b) for b in bad]}")
    return df[~df["Tahun"].isin(bad)].copy()

kec = drop_empty_years(kec, ["luas_panen", "produksi_kg"], "kecamatan")
nas = drop_empty_years(nas, ["luas_panen", "produksi_kg"], "nasional")

# ---------------------------------------------------------------- 6. MISSING VALUE
# Indikator missing disimpan SEBELUM imputasi ('-' bisa berarti tidak ada tanaman / belum dilaporkan)
for c in ["luas_panen", "produksi_kg"]:
    kec[f"{c}_missing"] = kec[c].isna().astype(int)
    nas[f"{c}_missing"] = nas[c].isna().astype(int)

def impute_series(df, keys, cols, strategy="interpolate"):
    """'none'        -> biarkan NaN
       'zero'        -> isi 0 (asumsi '-' = tidak ada produksi)
       'interpolate' -> interpolasi linear HANYA di antara dua titik valid (tanpa ekstrapolasi)"""
    df = df.sort_values(keys + ["Tahun"]).copy()
    for c in cols:
        if strategy == "zero":
            df[c] = df[c].fillna(0)
        elif strategy == "interpolate":
            df[c] = (df.groupby(keys)[c]
                     .transform(lambda s: s.interpolate(method="linear", limit_area="inside")))
    return df

kec = impute_series(kec, ["Kecamatan", "Tanaman"], ["luas_panen", "produksi_kg"], "interpolate")
nas = impute_series(nas, ["Tanaman"], ["luas_panen", "produksi_kg"], "interpolate")
print("\nSisa NaN kecamatan setelah imputasi:\n", kec[["luas_panen", "produksi_kg"]].isna().sum().to_string())

# ---------------------------------------------------------------- 7. VALIDASI LOGIKA
def validate(df, name):
    neg = (df[["luas_panen", "produksi_kg"]] < 0).sum().sum()
    a = df[(df["luas_panen"].fillna(0) == 0) & (df["produksi_kg"].fillna(0) > 0)]
    b = df[(df["produksi_kg"].fillna(0) == 0) & (df["luas_panen"].fillna(0) > 0)]
    print(f"\n[{name}] nilai negatif: {neg} | produksi>0 tapi luas=0/NaN: {len(a)} "
          f"| luas>0 tapi produksi=0/NaN: {len(b)}")

validate(kec, "kecamatan"); validate(nas, "nasional")

# Fitur turunan: produktivitas (kg per satuan luas)
for d in (kec, nas):
    d["produktivitas"] = np.where(d["luas_panen"] > 0, d["produksi_kg"] / d["luas_panen"], np.nan)

# Rekonsiliasi: jumlah kecamatan vs file total
rekon = (kec.groupby(["Tahun", "Tanaman"])[["luas_panen", "produksi_kg"]].sum(min_count=1)
         .join(nas.set_index(["Tahun", "Tanaman"])[["luas_panen", "produksi_kg"]],
               lsuffix="_sum_kec", rsuffix="_total", how="inner"))
rekon["rasio_prod"] = rekon["produksi_kg_sum_kec"] / rekon["produksi_kg_total"]
rekon["rasio_luas"] = rekon["luas_panen_sum_kec"] / rekon["luas_panen_total"]
print("\nRasio SUM(kecamatan) / total (ideal = 1.0):")
print(rekon[["rasio_prod", "rasio_luas"]].describe().loc[["mean", "min", "max"]].round(2).to_string())

# ---------------------------------------------------------------- 8. OUTLIER (FLAG, BUKAN HAPUS)
def flag_outlier_iqr(df, col, group="Tanaman", k=3.0):
    x = np.log1p(df[col])          # log: data sangat skewed (Kunyit jutaan vs Keji Beling puluhan)
    g = x.groupby(df[group])
    q1, q3 = g.transform(lambda s: s.quantile(.25)), g.transform(lambda s: s.quantile(.75))
    iqr = q3 - q1
    return ((x < q1 - k * iqr) | (x > q3 + k * iqr)).astype(int)

kec["outlier_luas"] = flag_outlier_iqr(kec, "luas_panen")
kec["outlier_prod"] = flag_outlier_iqr(kec, "produksi_kg")
print("\nJumlah outlier ter-flag:", kec[["outlier_luas", "outlier_prod"]].sum().to_dict())

# ---------------------------------------------------------------- 9. VISUALISASI
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
miss = kec.pivot_table(index="Kecamatan", columns="Tanaman",
                       values="luas_panen_missing", aggfunc="mean")
im = ax[0].imshow(miss, aspect="auto", cmap="Reds")
ax[0].set_title("Proporsi missing luas panen (sebelum imputasi)")
ax[0].set_xticks(range(miss.shape[1])); ax[0].set_xticklabels(miss.columns, rotation=90, fontsize=7)
ax[0].set_yticks(range(miss.shape[0])); ax[0].set_yticklabels(miss.index, fontsize=7)
plt.colorbar(im, ax=ax[0])

top = kec.groupby("Tanaman")["produksi_kg"].sum().nlargest(5).index
for t in top:
    s = kec[kec.Tanaman == t].groupby("Tahun")["produksi_kg"].sum()
    ax[1].plot(s.index, s.values, marker="o", label=t)
ax[1].set_yscale("log"); ax[1].set_title("Total produksi (kg) top 5 tanaman"); ax[1].legend()

ax[2].hist(np.log1p(kec["produksi_kg"].dropna()), bins=40)
ax[2].set_title("Distribusi log(1+produksi): sangat skewed")
plt.tight_layout()
plt.savefig(OUT_DIR / "eda_cleaning.png", dpi=150)
plt.show()

# ---------------------------------------------------------------- 10. SIMPAN
kec.to_csv(OUT_DIR / "biofarmaka_kecamatan_clean.csv", index=False)
nas.to_csv(OUT_DIR / "biofarmaka_nasional_clean.csv", index=False)
print("\nSelesai. kec:", kec.shape, "| nas:", nas.shape)
print("Output tersimpan di:", OUT_DIR)

File CSV tidak ditemukan. Lokasi yang dicari:
  - d:\SDT\Project Analitika Data\AgriDash-0.1\notebooks
  - d:\SDT\Project Analitika Data\AgriDash-0.1\notebooks\data
  - d:\SDT\Project Analitika Data\AgriDash-0.1\notebooks
  - d:\SDT\Project Analitika Data\AgriDash-0.1\notebooks\data
Isi folder skrip: ['.env.example', 'Cleaning_Biofarmaka.ipynb', 'Cleaning_Hortikultura.ipynb', 'cleaning_perkebunan.ipynb', 'cleaning_tanaman_pangan.ipynb', 'gitignore', 'Pembagian_tanaman_pangan.ipynb', 'requirements.txt']


FileNotFoundError: Taruh keempat CSV di folder yang sama dengan skrip, atau di subfolder 'data'.